# **(3)** Transfer learning
  
The pre-trained 4-head backbone is reused as-is; only the **output layer is swapped** from 4 → 3 heads.

```
pretraining_multitask_model.pt
        │
        ├─ message_passing.*   → FROZEN      (loaded from PT)
        ├─ FFN hidden layers   → TRAINABLE   (loaded from PT, good initialisation)
        └─ FFN output layer    → REPLACED    (4 heads → 3 heads, re-initialised)
                │
                ▼
        [purity,  camsol_score,  revised_50hemo]
```

In [ ]:
pip install chemprop lightning rdkit pandas matplotlib

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    if not os.path.exists("IX_technical_turorial2026"):
        !git clone https://github.com/rbirolo/IX_technical_turorial2026.git
    
    %cd IX_technical_turorial2026

In [ ]:
import warnings, pickle
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightning as pl
from scipy import stats as sp_stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

from chemprop.data import (
    MoleculeDatapoint, MoleculeDataset, build_dataloader,
    make_split_indices, split_data_by_indices,
)
from chemprop.models import MPNN
from chemprop.nn import BondMessagePassing, MeanAggregation, RegressionFFN, RMSE, MAE
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

## Configuration

Architecture Should match the pre-trained model (except output_layer : `n_tasks`).


In [ ]:
FT_data_file = "data/experimental_dataset.csv"
MT_output    = Path("models/chemprop_pretraining")    # PT checkpoint
FT_output    = Path("chemprop_ft_model")    # FT weights saved here
FT_targets = ["purity", "camsol_score", "revised_50hemo"]

# Architecture (matching pre-training)
MP_DEPTH   = 3
MP_DIM     = 300
FFN_LAYERS = 2
FFN_DIM    = 300
DROPOUT    = 0.2

# Training
MAX_EPOCHS = 50
BATCH_SIZE = 64
INIT_LR    = 1e-4
MAX_LR     = 1e-3
FINAL_LR   = 1e-4
WARMUP_EPS = 2
SEED       = 42
SPLIT_SIZES = (0.70, 0.15, 0.15)

## Load 'experimental' data

Standard multi-task loading — `y` shape `(3,)`, NaN kept and masked in loss.


In [ ]:
def load_dataset(path, targets):
    df = pd.read_csv(path)
    for t in targets:
        df[t] = pd.to_numeric(df[t], errors="coerce")
    df = df.dropna(subset=["SMILES"]).copy()

    mols, valid_idx = [], []
    for i, smi in enumerate(df["SMILES"]):
        mol = Chem.MolFromSmiles(str(smi))
        if mol is not None:
            mols.append(mol); valid_idx.append(i)
        else:
            print(f"Invalid SMILES (row {i}): {str(smi)[:80]}…")

    df  = df.iloc[valid_idx].reset_index(drop=True)
    ys  = df[targets].values.astype(float)
    return mols, ys, df

mols, ys, df_clean = load_dataset(FT_data_file, FT_targets)
N = len(mols)

print(f"\n{'Target':<22} {'non-NaN':>8} {'mean':>9} {'std':>8} {'min':>8} {'max':>8}")
print("─" * 66)
for i, t in enumerate(FT_targets):
    v = ys[:, i][~np.isnan(ys[:, i])]
    print(f"  {t:<20} {len(v):>8} {v.mean():>9.3f} {v.std():>8.3f} {v.min():>8.3f} {v.max():>8.3f}")

## Defining functions

In [ ]:
def make_loaders(datapoints):
    train, val, test = split_data_by_indices(datapoints, train_idxs, val_idxs, test_idxs)
    train = [dp for sub in train for dp in sub]
    val   = [dp for sub in val   for dp in sub]
    test  = [dp for sub in test  for dp in sub]

    train_ds = MoleculeDataset(train)
    val_ds   = MoleculeDataset(val)
    test_ds  = MoleculeDataset(test)

    scaler = train_ds.normalize_targets()
    val_ds.normalize_targets(scaler)

    train_load = build_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_load   = build_dataloader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_load  = build_dataloader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_load, val_load, test_load, scaler


def make_mpnn(n_tasks: int) -> MPNN:
    mp  = BondMessagePassing(depth=MP_DEPTH, d_h=MP_DIM, dropout=DROPOUT)
    agg = MeanAggregation()
    ffn = RegressionFFN(n_tasks=n_tasks, input_dim=mp.output_dim,
                        hidden_dim=FFN_DIM, n_layers=FFN_LAYERS, dropout=DROPOUT)
    return MPNN(mp, agg, ffn, metrics=[RMSE(), MAE()],
                init_lr=INIT_LR, max_lr=MAX_LR, final_lr=FINAL_LR,
                warmup_epochs=WARMUP_EPS)

def run_trainer(model, train_load, val_load):
    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS, logger=False,
        enable_checkpointing=False, enable_progress_bar=True, accelerator="auto",
    )
    trainer.fit(model, train_load, val_load)
    return trainer

def run_trainer_earlystopping (model, train_load, val_load): #adding early stopping
  early_stop = EarlyStopping(
      monitor="val_loss",
      patience=10,
      mode="min",
      verbose=True,
      )
  checkpoint = ModelCheckpoint(
      monitor="val_loss",
      mode="min",
      save_top_k=1,
      filename="best_model",
      )
  trainer = pl.Trainer(
      max_epochs=500,
      logger=False,
      enable_checkpointing=True,
      enable_progress_bar=True,
      accelerator="auto",
      callbacks=[early_stop, checkpoint],
      )
  trainer.fit(model, train_load, val_load)
  print(f"\nStopped at epoch {trainer.current_epoch} / {MAX_EPOCHS}")
  print(f"Best val_loss: {checkpoint.best_model_score:.5f}")

  #Load best weights back into the model
  best_ckpt = torch.load(
      checkpoint.best_model_path,
      weights_only=False,
      )
  model.load_state_dict(best_ckpt["state_dict"])
  return trainer


def evaluate_mt(model, test_loader, scaler, target_names):
    model.eval()
    all_p, all_gt = [], []
    with torch.no_grad():
        for batch in test_loader:
            bmg, V_d, X_d, targets, *_ = batch
            all_p.append(model(bmg, V_d, X_d).numpy())
            all_gt.append(targets.numpy())

    preds = scaler.inverse_transform(np.concatenate(all_p))
    gts   = np.concatenate(all_gt)

    out = {}
    for i, t in enumerate(target_names):
        mask = ~np.isnan(gts[:, i])
        if mask.sum() < 2: continue
        gt, pr = gts[mask, i], preds[mask, i]
        out[t] = {
            "gt": gt, "pr": pr,
            "rmse"      : float(np.sqrt(mean_squared_error(gt, pr))),
            "mae"       : float(mean_absolute_error(gt, pr)),
            "pearson_r" : float(sp_stats.pearsonr(gt, pr)[0]),
            "spearman_r": float(pd.Series(gt).corr(pd.Series(pr), method="spearman")),
            "n"         : int(mask.sum()),
        }
    return out

## Data Split

In [ ]:
pl.seed_everything(SEED)

train_idxs, val_idxs, test_idxs = make_split_indices(
    mols, split="random", sizes=SPLIT_SIZES, seed=SEED
)
_ti, _vi, _tei = train_idxs[0], val_idxs[0], test_idxs[0]
print(f"Train : {len(_ti):>5}  ({len(_ti)/N*100:.1f}%)")
print(f"Val   : {len(_vi):>5}  ({len(_vi)/N*100:.1f}%)")
print(f"Test  : {len(_tei):>5}  ({len(_tei)/N*100:.1f}%)")

## Load Pre-Trained Weights → Build 3-Head FT Model

Three groups of parameters:

| Layer | Action |
|---|---|
| `message_passing.*` | **Loaded + frozen** |
| `predictor.ffn.0.0.*` / `ffn.1.2.*` | **Loaded, trainable** (hidden layers) |
| `predictor.ffn.2.2.*` | **Discarded** — shape `[4,300]` incompatible with 3 heads → re-initialised |


In [ ]:
# Load checkpoints
ckpt = torch.load(MT_output / "pretraining_multitask_model.pt", weights_only=True)
n_pt = ckpt["predictor.ffn.2.2.bias"].shape[0]
print(f"Checkpoint: {n_pt} PT heads, {len(ckpt)} keys")

# Build a new 3-head model
ft_model = make_mpnn(n_tasks=len(FT_targets))   # 3 heads

# Transfer backbone + FFN hidden layers (skip output layer)
partial_weights = {
    k: v for k, v in ckpt.items()
    if k.startswith("message_passing")
    or k in ("predictor.ffn.0.0.weight", "predictor.ffn.0.0.bias",
             "predictor.ffn.1.2.weight", "predictor.ffn.1.2.bias")
}
missing, unexpected = ft_model.load_state_dict(partial_weights, strict=False)
print(f"Transferred : {len(partial_weights)} tensors")
print(f"Missing     : {len(missing)}  (output layer, randomly initialised)")

# Freeze backbone
for name, param in ft_model.named_parameters():
    if name.startswith("message_passing"):
        param.requires_grad = False

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in ft_model.parameters() if not p.requires_grad)
print(f"\nTrainable : {trainable:,} params")
print(f"Frozen : {frozen:,} params  (backbone)")

In [ ]:
#@title Optional: Experiments with different frozen layers

# Pretrained weights loaded (initialisation) but verything is trainable
for param in ft_model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in ft_model.parameters() if not p.requires_grad)

print(f"\nTrainable : {trainable:,} params")
print(f"Frozen    : {frozen:,} params")

# Freeze message passing + FFN
# Only the output layer remains trainable

for name, param in ft_model.named_parameters():
    if name.startswith("message_passing") or name.startswith("predictor.ffn.0") or name.startswith("predictor.ffn.1"):
        param.requires_grad = False
    else:
        param.requires_grad = True

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in ft_model.parameters() if not p.requires_grad)

print(f"Trainable : {trainable:,} params  (output layer)")
print(f"Frozen    : {frozen:,} params  (message_passing + FFN)")

## Transfer learning (fine-tuning/ feature extraction)
Fine-tuning: pre-trained backbone updated + new task head.

Feature extraction: pre-trained backbone **frozen** + new task head trained.


In [ ]:
ft_datapoints = [
    MoleculeDatapoint(mol=m, y=y.flatten())
    for m, y in zip(mols, ys)
]
print(f"{len(ft_datapoints)} datapoints  |  y shape: ({len(FT_targets)},)")

ft_train_load, ft_val_load, ft_test_load, ft_scaler = make_loaders(ft_datapoints)

#run_trainer(ft_model, ft_train_load, ft_val_load) #50 epochs for comparison with previous notebooks --> model saved as
run_trainer_earlystopping(ft_model, ft_train_load, ft_val_load)

## 7 · Save Fine-Tuned Model

In [ ]:
FT_output.mkdir(parents=True, exist_ok=True)
torch.save(ft_model.state_dict(), FT_output / "finetuned_model_earlystopping.pt")
with open(FT_output / "finetuned_model_earlystopping.pkl", "wb") as f:
    pickle.dump(ft_scaler, f)

## 8 · Test Set Results

Performance on the 3 experimental targets.


In [ ]:
ft_metrics = evaluate_mt(ft_model, ft_test_load, ft_scaler, FT_targets)

print(f"{'Target':<22} {'RMSE':>8} {'MAE':>8} {'Pearson':>9} {'Spearman':>10} {'N':>6}")
print("─" * 76)
for t in FT_targets:
    if t not in ft_metrics:
        print(f"  {t:<20}  — no data"); continue
    m = ft_metrics[t]
    print(f"  {t:<20}  {m['rmse']:>8.4f} {m['mae']:>8.4f}"
          f" {m['pearson_r']:>9.4f} {m['spearman_r']:>10.4f} {m['n']:>6}")

In [ ]:
fig, axes = plt.subplots(1, len(FT_targets), figsize=(5.5 * len(FT_targets), 5))
colors = ["#2196F3", "#4CAF50", "#E91E63"]

for ax, t, c in zip(axes, FT_targets, colors):
    if t not in ft_metrics:
        ax.set_visible(False); continue
    m  = ft_metrics[t]
    lo = min(m["gt"].min(), m["pr"].min()) * 0.95
    hi = max(m["gt"].max(), m["pr"].max()) * 1.05

    ax.plot([lo, hi], [lo, hi], "k--", lw=1, zorder=1)
    ax.scatter(m["gt"], m["pr"], color=c, edgecolors="white", s=60,
               linewidths=0.5, alpha=0.85, zorder=2,
               label=f"RMSE={m['rmse']:.3f}\nPearson r={m['pearson_r']:.3f}\nN={m['n']}")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_title(t, fontsize=12, fontweight="bold")
    ax.set_xlabel("ground truth"); ax.set_ylabel("predicted")
    ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

plt.suptitle("Fine-tuned model — test set", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## Model comparison
Early stopping and MT_50epochs.

In [ ]:
# Load model 1 - early stopping
model1 = make_mpnn(n_tasks=len(FT_targets))
ckpt1 = torch.load("models/chemprop_mt/multitask_model.pt", weights_only=True)
model1.load_state_dict(ckpt1)
model1.eval()

# Load model 2 - 50 epochs for comparison with ST and MT
model2 = make_mpnn(n_tasks=len(FT_targets))
ckpt2 = torch.load("models/chemprop_ft_model/finetuned_model_50epochs.pt", weights_only=True)
model2.load_state_dict(ckpt2)
model2.eval()

# Load model 3 - MT 50 epochs 
model3 = make_mpnn(n_tasks=len(FT_targets))
ckpt3 = torch.load("models/chemprop_ft_model/finetuned_model_earlystopping.pt", weights_only=True)
model3.load_state_dict(ckpt3)
model3.eval()

# Evaluate both on the same test set
metrics1 = evaluate_mt(model1, ft_test_load, ft_scaler, FT_targets)
metrics2 = evaluate_mt(model2, ft_test_load, ft_scaler, FT_targets)
metrics3 = evaluate_mt(model3, ft_test_load, ft_scaler, FT_targets)

In [ ]:
fig, axes = plt.subplots(
    1, len(FT_targets),
    figsize=(5.5 * len(FT_targets), 5)
)

colors = ["#2196F3", "#4CAF50", "#E91E63"]

for ax, t in zip(axes, FT_targets):

    if t not in metrics1 or t not in metrics2 or t not in metrics3:
        ax.set_visible(False)
        continue

    m1 = metrics1[t]
    m2 = metrics2[t]
    m3 = metrics3[t]

    lo = min(
        m1["gt"].min(), m1["pr"].min(),
        m2["gt"].min(), m2["pr"].min(),
        m3["gt"].min(), m3["pr"].min(),
    ) * 0.95

    hi = max(
        m1["gt"].max(), m1["pr"].max(),
        m2["gt"].max(), m2["pr"].max(),
        m3["gt"].max(), m3["pr"].max(),
    ) * 1.05

    ax.plot([lo, hi], [lo, hi], "k--", lw=1, zorder=1)

    ax.scatter(m1["gt"], m1["pr"], color=colors[0], edgecolors="white",
               s=60, linewidths=0.5, alpha=0.65, zorder=2,
               label=f"MT_50epochs\nPearson r={m1['pearson_r']:.3f}")
    ax.scatter(m2["gt"], m2["pr"], color=colors[1], edgecolors="white",
               s=60, linewidths=0.5, alpha=0.65, zorder=3,
               label=f"PT_50epochs\nPearson r={m2['pearson_r']:.3f}")
    ax.scatter(m3["gt"], m3["pr"], color=colors[2], edgecolors="white",
               s=60, linewidths=0.5, alpha=0.65, zorder=4,
               label=f"PT_earlystop\nPearson r={m3['pearson_r']:.3f}")


    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_title(t, fontsize=12, fontweight="normal")
    ax.set_xlabel("y_value"); ax.set_ylabel("predicted")
    ax.legend(fontsize=8); ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()